In [27]:
import numpy as np
import torch
import tensorflow as tf

In [23]:
class Neuron:
    def __init__(self, nin):
        self.w = [torch.Tensor([np.random.uniform(-1,1)]) for _ in range(nin)]
        self.b = torch.Tensor([np.random.uniform(-1,1)])
    
    def __call__(self, x):
        # z = X @ W + b
        z = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = z.tanh()
        return out
    
    def parameters(self):
        return self.w + [self.b]

In [24]:
x = [3.1, 2.5, 6.5]
n = Neuron(len(x))
n(x)

tensor([0.9825])

In [26]:
len(n.parameters())

4

In [29]:
class Layer:
    def __init__(self, nin, n_neurons):
        self.neurons = [Neuron(nin) for _ in range(n_neurons)]
        
    def __call__(self, x):
        out = [neuron(x) for neuron in self.neurons]
        return out if len(out) > 1 else out[0]

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
    

In [30]:
l = Layer(len(x), 4)
l(x)

[tensor([0.9983]), tensor([1.]), tensor([-0.6102]), tensor([0.9974])]

In [ ]:
# tensorflow implementation -> this layer is called a Dense layer in tf
class myDenseLayer(tf.keras.layers.Layer):
    def __init__(self, input_dim, out_dim):
        super(myDenseLayer, self).__init__()
        self.w = self.add_weight(shape=(input_dim, out_dim), initializer='random_normal', trainable=True)
        self.b = self.add_weight(shape=(1, out_dim), initializer='random_normal', trainable=True)
    
    def __call__(self, inputs):
        z = tf.matmul(inputs, self.w) + self.b
        out = tf.nn.sigmoid(z)
        return out

In [ ]:
# torch implementation -> layer is called Linear layer in torch
class myDenseLayer(tf.keras.layers.Layer):
    def __init__(self, input_dim, out_dim):
        self.w = torch.nn.Parameter(torch.randn((input_dim, out_dim)), requires_grad=True)
        self.b = torch.nn.Parameter(torch.randn(1, out_dim), requires_grad=True)
     
    def forward(self, inputs):
        z = torch.matmul(inputs, self.w) + self.b
        out = torch.nn.Sigmoid(z)
        return out

In [ ]:
# tf gradient descent implementation

weights = tf.Variable([tf.random.normal()])

while True:
    with tf.GradientTape() as g:
        loss = compute_loss(weights)
    gradients = g.gradient(loss, weights)
    # weights -= (learning_rate * gradients)
    weights = weights - (learning_rate * gradients)
    

# Why can we use larger learning rstes when we optimize using mini-batch GD?

in mini batch gradient descent, the graident is averaged over a batch of training examples, say 32, which gives us a better (less noisy) idea about the gradient and the surface of the loss in that region, whihc means we can increase our step size because we have a better view of how the loss landscape looks.

On the other hand, when we use Stochastic Gradient Descent, we take the gradient based on ONE training example. This means that the graidnet is noisy and not representive of the whole distribution of grads. therefore, it is better to use a smaller `lr` here so that we step slowly and carefully through the landscape of the loss

In [ ]:
import tensorflow as tf
class myRnn(tf.keras.layers.Layer):
    def __init__(self, in_dim, rnn_units, out_dim):
        super().__init__()
        self.in_dim = in_dim
        self.rnn_units = rnn_units
        self.out_dim = out_dim
        
        # weight matrics
        self.w_xh = self.add_weight([rnn_units, in_dim])
        self.w_hh = self.add_weight([rnn_units, rnn_units])
        self.w_hy = self.add_weight([out_dim, rnn_units])
        
        # hidden state vector
        self.h = tf.zeros([rnn_units, 1])
    
    def call(self, x):
        # perfprm the forawrd pass
        self.h = (self.w_hh @ self.h) + (self.w_xh @ x)
        self.h = tf.tanh(self.h)
        y_hat = self.w_hy @ self.h
        
        # remember that RNN returns predictions dn updated hidden state
        return y_hat, self.h

In [ ]:
my_rnn = myRnn(in_dim=10, rnn_units=20, out_dim=5)
x = tf.random.normal([10, 1])
y_hat, h = my_rnn(x)

In [8]:
y_hat, h

(<tf.Tensor: shape=(5, 1), dtype=float32, numpy=
 array([[0.710461  ],
        [0.2474893 ],
        [0.17909986],
        [1.1027191 ],
        [0.13213332]], dtype=float32)>,
 <tf.Tensor: shape=(20, 1), dtype=float32, numpy=
 array([[ 0.2559787 ],
        [ 0.00816224],
        [ 0.73199075],
        [ 0.08982426],
        [-0.17184046],
        [-0.47057074],
        [ 0.24888411],
        [ 0.05202586],
        [ 0.51339877],
        [-0.06532142],
        [ 0.6450118 ],
        [-0.2366596 ],
        [-0.03411673],
        [-0.39986286],
        [-0.28298312],
        [-0.27836013],
        [ 0.18511496],
        [-0.6114852 ],
        [-0.49700898],
        [ 0.22587106]], dtype=float32)>)

In [13]:
import tensorflow as tf

class MyRNNCell(tf.keras.layers.Layer):
    def __init__(self, in_dim, rnn_units, out_dim):
        super().__init__()
        self.w_xh = self.add_weight(shape=(in_dim, rnn_units))
        self.w_hh = self.add_weight(shape=(rnn_units, rnn_units))
        self.b_h = self.add_weight(shape=(rnn_units,), initializer="zeros")

        self.w_hy = self.add_weight(shape=(rnn_units, out_dim))
        self.b_y = self.add_weight(shape=(out_dim,), initializer="zeros")

    def call(self, x, h_prev):
        h = tf.tanh(tf.matmul(x, self.w_xh) + tf.matmul(h_prev, self.w_hh) + self.b_h)
        y_hat = tf.matmul(h, self.w_hy) + self.b_y
        return y_hat, h

In [15]:
x = tf.random.normal([5, 1, 10])
h_prev = tf.zeros([5, 1, 20])
my_rnn_cell = MyRNNCell(in_dim=10, rnn_units=20, out_dim=5)
y_hat, h = my_rnn_cell(x, h_prev)
y_hat.shape, h.shape

(TensorShape([5, 1, 5]), TensorShape([5, 1, 20]))